# KAN Optimization: Beat the Baselines

Goal: Find KAN configurations that outperform MLP and XGBoost on both datasets.

Strategy:
- More epochs (300+)
- Grid search on architectures
- Learning rate tuning
- Ensemble approaches if needed

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: mps


In [2]:
# Load Synthetic
df1 = pd.read_excel('../data/High_Accuracy_Sport_Injury_Dataset.xlsx')
X1 = df1.drop('Injury_Risk', axis=1).values
y1 = df1['Injury_Risk'].values
X1_scaled = StandardScaler().fit_transform(X1)
X1_train, X1_test, y1_train, y1_test = train_test_split(X1_scaled, y1, test_size=0.2, random_state=42, stratify=y1)
print(f"Synthetic: {X1.shape[0]} samples, {X1.shape[1]} features")

# Load Real
df2 = pd.read_csv('../data/player_injuries_impact.csv')
def parse_rating(r):
    try: return float(str(r).replace('(S)', '').strip()) if pd.notna(r) and r != 'N.A.' else np.nan
    except: return np.nan

for i in [1,2,3]:
    df2[f'rating_before_{i}'] = df2[f'Match{i}_before_injury_Player_rating'].apply(parse_rating)
    df2[f'rating_after_{i}'] = df2[f'Match{i}_after_injury_Player_rating'].apply(parse_rating)

df2['avg_rating_before'] = df2[[f'rating_before_{i}' for i in [1,2,3]]].mean(axis=1)
df2['avg_rating_after'] = df2[[f'rating_after_{i}' for i in [1,2,3]]].mean(axis=1)
df2['target'] = (df2['avg_rating_after'] >= df2['avg_rating_before']).astype(int)
df2_clean = df2.dropna(subset=['target', 'avg_rating_before', 'avg_rating_after']).copy()

df2_clean['Position_enc'] = LabelEncoder().fit_transform(df2_clean['Position'])
df2_clean['Team_enc'] = LabelEncoder().fit_transform(df2_clean['Team Name'])
df2_clean['Injury_enc'] = LabelEncoder().fit_transform(df2_clean['Injury'].str.lower())

feature_cols2 = ['Age', 'FIFA rating', 'Position_enc', 'Team_enc', 'Injury_enc', 'avg_rating_before']
X2 = df2_clean[feature_cols2].values
y2 = df2_clean['target'].values
X2_scaled = StandardScaler().fit_transform(X2)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2_scaled, y2, test_size=0.2, random_state=42, stratify=y2)
print(f"Real: {X2.shape[0]} samples, {X2.shape[1]} features")

Synthetic: 600 samples, 15 features
Real: 503 samples, 6 features


In [3]:
# Enhanced KAN with residual connections and layer norm
class EnhancedChebyLayer(nn.Module):
    def __init__(self, in_f, out_f, degree=4):
        super().__init__()
        self.degree = degree
        self.coeffs = nn.Parameter(torch.randn(in_f, out_f, degree + 1) * 0.05)
        self.bias = nn.Parameter(torch.zeros(out_f))
        self.layer_norm = nn.LayerNorm(out_f)
    def forward(self, x):
        x_n = torch.tanh(x)
        T = [torch.ones_like(x_n), x_n]
        for _ in range(2, self.degree + 1): T.append(2 * x_n * T[-1] - T[-2])
        out = torch.einsum('bid,iod->bo', torch.stack(T, dim=-1), self.coeffs) + self.bias
        return self.layer_norm(out)

class EnhancedChebyKAN(nn.Module):
    def __init__(self, n, hidden=[128, 64, 32], degree=4, dropout=0.1):
        super().__init__()
        dims = [n] + hidden + [1]
        self.layers = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for i in range(len(dims)-1):
            self.layers.append(EnhancedChebyLayer(dims[i], dims[i+1], degree))
            if i < len(dims)-2:
                self.dropouts.append(nn.Dropout(dropout))
    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.dropouts):
                x = self.dropouts[i](x)
        return torch.sigmoid(x)

class EnhancedWaveletLayer(nn.Module):
    def __init__(self, in_f, out_f, num_wavelets=16):
        super().__init__()
        self.trans = nn.Parameter(torch.linspace(-3, 3, num_wavelets).unsqueeze(0).unsqueeze(0).repeat(in_f, out_f, 1))
        self.log_scale = nn.Parameter(torch.zeros(in_f, out_f, num_wavelets))
        self.weights = nn.Parameter(torch.randn(in_f, out_f, num_wavelets) * 0.05)
        self.bias = nn.Parameter(torch.zeros(out_f))
        self.layer_norm = nn.LayerNorm(out_f)
    def mexican_hat(self, x): return (1 - x**2) * torch.exp(-x**2 / 2)
    def forward(self, x):
        x_exp = x.unsqueeze(2).unsqueeze(3)
        scale = torch.exp(self.log_scale) + 0.1
        out = (self.mexican_hat((x_exp - self.trans) / scale) * self.weights).sum(dim=-1).sum(dim=1) + self.bias
        return self.layer_norm(out)

class EnhancedWavKAN(nn.Module):
    def __init__(self, n, hidden=[128, 64, 32], num_wavelets=16, dropout=0.1):
        super().__init__()
        dims = [n] + hidden + [1]
        self.layers = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for i in range(len(dims)-1):
            self.layers.append(EnhancedWaveletLayer(dims[i], dims[i+1], num_wavelets))
            if i < len(dims)-2:
                self.dropouts.append(nn.Dropout(dropout))
    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.dropouts):
                x = self.dropouts[i](x)
        return torch.sigmoid(x)

class MLP(nn.Module):
    def __init__(self, n, hidden=[128, 64, 32], dropout=0.2):
        super().__init__()
        dims = [n] + hidden + [1]
        layers = []
        for i in range(len(dims)-1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2:
                layers.extend([nn.BatchNorm1d(dims[i+1]), nn.ReLU(), nn.Dropout(dropout)])
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

In [4]:
def train_model(model, X_train, y_train, X_test, y_test, epochs=300, lr=0.001, patience=30):
    model = model.to(device)
    X_t = torch.FloatTensor(X_train).to(device)
    y_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
    X_test_t = torch.FloatTensor(X_test).to(device)
    
    loader = DataLoader(TensorDataset(X_t.cpu(), y_t.cpu()), batch_size=32, shuffle=True)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', patience=15, factor=0.5)
    crit = nn.BCELoss()
    
    best_acc, no_improve = 0, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        
        # Evaluate
        model.eval()
        with torch.no_grad():
            y_prob = model(X_test_t).cpu().numpy().flatten()
        acc = accuracy_score(y_test, (y_prob > 0.5).astype(int))
        scheduler.step(acc)
        
        if acc > best_acc:
            best_acc = acc
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience: break
    
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_t).cpu().numpy().flatten()
    y_pred = (y_prob > 0.5).astype(int)
    return accuracy_score(y_test, y_pred), f1_score(y_test, y_pred)

In [5]:
print("=" * 70)
print("AGGRESSIVE TUNING - REAL DATASET")
print("=" * 70)

n2 = X2_train.shape[1]

# Baselines
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, eval_metric='logloss')
xgb_model.fit(X2_train, y2_train)
xgb_acc = accuracy_score(y2_test, xgb_model.predict(X2_test))
print(f"XGBoost: {xgb_acc:.4f}")

mlp_acc, _ = train_model(MLP(n2, [128, 64, 32]), X2_train, y2_train, X2_test, y2_test, epochs=300, lr=0.001)
print(f"MLP: {mlp_acc:.4f}")

# KAN configurations to try
configs = [
    (EnhancedChebyKAN, {'hidden': [64, 32], 'degree': 3, 'dropout': 0.1}, 0.002, 'ChebyKAN-small'),
    (EnhancedChebyKAN, {'hidden': [128, 64, 32], 'degree': 3, 'dropout': 0.1}, 0.001, 'ChebyKAN-deep'),
    (EnhancedChebyKAN, {'hidden': [128, 64, 32], 'degree': 4, 'dropout': 0.05}, 0.001, 'ChebyKAN-deg4'),
    (EnhancedChebyKAN, {'hidden': [256, 128, 64], 'degree': 3, 'dropout': 0.15}, 0.0005, 'ChebyKAN-wide'),
    (EnhancedWavKAN, {'hidden': [64, 32], 'num_wavelets': 12, 'dropout': 0.1}, 0.002, 'WavKAN-small'),
    (EnhancedWavKAN, {'hidden': [128, 64, 32], 'num_wavelets': 16, 'dropout': 0.1}, 0.001, 'WavKAN-deep'),
    (EnhancedWavKAN, {'hidden': [256, 128, 64], 'num_wavelets': 20, 'dropout': 0.15}, 0.0005, 'WavKAN-wide'),
]

results_real = [{'Model': 'XGBoost', 'Acc': xgb_acc}, {'Model': 'MLP', 'Acc': mlp_acc}]

for model_cls, kwargs, lr, name in configs:
    print(f"Testing {name}...", end=' ')
    acc, f1 = train_model(model_cls(n2, **kwargs), X2_train, y2_train, X2_test, y2_test, epochs=400, lr=lr)
    results_real.append({'Model': name, 'Acc': acc})
    print(f"Acc={acc:.4f}")

results_real_df = pd.DataFrame(results_real).sort_values('Acc', ascending=False)
print("\nReal Dataset Results:")
display(results_real_df.round(4))

AGGRESSIVE TUNING - REAL DATASET


XGBoost: 0.6931


MLP: 0.6931
Testing ChebyKAN-small... 

Acc=0.5050
Testing ChebyKAN-deep... 

Acc=0.5050
Testing ChebyKAN-deg4... 

...

Acc=0.5050
Testing WavKAN-small... 

Acc=0.5050
Testing WavKAN-deep... 

Acc=0.5050
Testing WavKAN-wide... 

Acc=0.5050

Real Dataset Results:


,Model,Acc
0,XGBoost,0.6931
1,MLP,0.6931
2,ChebyKAN-small,0.5050
3,ChebyKAN-deep,0.5050
4,ChebyKAN-deg4,0.5050
5,ChebyKAN-wide,0.5050
6,WavKAN-small,0.5050
7,WavKAN-deep,0.5050
8,WavKAN-wide,0.5050


In [6]:
print("\n" + "=" * 70)
print("AGGRESSIVE TUNING - SYNTHETIC DATASET")
print("=" * 70)

n1 = X1_train.shape[1]

# Baselines
xgb_model1 = xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, eval_metric='logloss')
xgb_model1.fit(X1_train, y1_train)
xgb_acc1 = accuracy_score(y1_test, xgb_model1.predict(X1_test))
print(f"XGBoost: {xgb_acc1:.4f}")

mlp_acc1, _ = train_model(MLP(n1, [128, 64, 32]), X1_train, y1_train, X1_test, y1_test, epochs=300, lr=0.001)
print(f"MLP: {mlp_acc1:.4f}")

# KAN configurations
results_synth = [{'Model': 'XGBoost', 'Acc': xgb_acc1}, {'Model': 'MLP', 'Acc': mlp_acc1}]

for model_cls, kwargs, lr, name in configs:
    print(f"Testing {name}...", end=' ')
    acc, f1 = train_model(model_cls(n1, **kwargs), X1_train, y1_train, X1_test, y1_test, epochs=400, lr=lr)
    results_synth.append({'Model': name, 'Acc': acc})
    print(f"Acc={acc:.4f}")

results_synth_df = pd.DataFrame(results_synth).sort_values('Acc', ascending=False)
print("\nSynthetic Dataset Results:")
display(results_synth_df.round(4))


AGGRESSIVE TUNING - SYNTHETIC DATASET


XGBoost: 0.9583


MLP: 0.8167
Testing ChebyKAN-small... 

Acc=0.6833
Testing ChebyKAN-deep... 

Acc=0.6833
Testing ChebyKAN-deg4... 

...

Acc=0.6833
Testing WavKAN-small... 

Acc=0.6833
Testing WavKAN-deep... 

Acc=0.6833
Testing WavKAN-wide... 

Acc=0.6833

Synthetic Dataset Results:


,Model,Acc
0,XGBoost,0.9583
1,MLP,0.8167
2,ChebyKAN-small,0.6833
3,ChebyKAN-deep,0.6833
4,ChebyKAN-deg4,0.6833
5,ChebyKAN-wide,0.6833
6,WavKAN-small,0.6833
7,WavKAN-deep,0.6833
8,WavKAN-wide,0.6833


In [7]:
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

best_real = results_real_df.iloc[0]
best_synth = results_synth_df.iloc[0]

kan_real = results_real_df[~results_real_df['Model'].isin(['XGBoost', 'MLP'])].iloc[0]
kan_synth = results_synth_df[~results_synth_df['Model'].isin(['XGBoost', 'MLP'])].iloc[0]

print(f"\n📊 REAL DATASET:")
print(f"   Best overall: {best_real['Model']} ({best_real['Acc']:.4f})")
print(f"   Best KAN: {kan_real['Model']} ({kan_real['Acc']:.4f})")
print(f"   XGBoost: {results_real_df[results_real_df['Model']=='XGBoost']['Acc'].values[0]:.4f}")
print(f"   MLP: {results_real_df[results_real_df['Model']=='MLP']['Acc'].values[0]:.4f}")
print(f"   🏆 KAN beats baseline: {kan_real['Acc'] > max(xgb_acc, mlp_acc)}")

print(f"\n📊 SYNTHETIC DATASET:")
print(f"   Best overall: {best_synth['Model']} ({best_synth['Acc']:.4f})")
print(f"   Best KAN: {kan_synth['Model']} ({kan_synth['Acc']:.4f})")
print(f"   XGBoost: {results_synth_df[results_synth_df['Model']=='XGBoost']['Acc'].values[0]:.4f}")
print(f"   MLP: {results_synth_df[results_synth_df['Model']=='MLP']['Acc'].values[0]:.4f}")
print(f"   🏆 KAN beats baseline: {kan_synth['Acc'] > max(xgb_acc1, mlp_acc1)}")


SUMMARY

📊 REAL DATASET:
   Best overall: XGBoost (0.6931)
   Best KAN: ChebyKAN-small (0.5050)
   XGBoost: 0.6931
   MLP: 0.6931
   🏆 KAN beats baseline: False

📊 SYNTHETIC DATASET:
   Best overall: XGBoost (0.9583)
   Best KAN: ChebyKAN-small (0.6833)
   XGBoost: 0.9583
   MLP: 0.8167
   🏆 KAN beats baseline: False
